In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [3]:
#Q1:
print(len(documents)) #number of lessons : 72
print(documents[0])

72
{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a s

In [4]:
#Q2
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
question="How does the agentic loop keep calling the model until it stops?"
result=index.search(question)
print(result[0]["filename"]) #Q2:01-agentic-rag/lessons/14-agentic-loop.md

01-agentic-rag/lessons/14-agentic-loop.md


In [5]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
import os

from rag_helper import RAGBase
openai_client =OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    course=result[0]["filename"]
)
ans=assistant.rag("How does the agentic loop keep calling the model until it stops?")
print(ans["response"])
print("input tockens :",ans["input_tokens"])   
#I'm using groq i got that i requeted 7183 and the limit is 6000 , I'm not sure but I think 
#what I've done at first was bettet, using a filter with the exact course name retriver by search on same querry so I got 2352 instead .

The agentic loop keeps calling the model until it stops by using a simple while loop in the `agent_loop` function. Here's the relevant code snippet:

```python
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTAN

In [7]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks) #"Q4 295 chunks"

295

In [8]:
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunked_index.fit(chunks)
result=chunked_index.search(question)
assistant = RAGBase(
    index=chunked_index,
    llm_client=openai_client,
    course=result[0]["filename"]
)
ans=assistant.rag("How does the agentic loop keep calling the model until it stops?")
print("input tockens :",ans["input_tokens"])  #2233 when I send the specific course
#using my mthod I think chunking wouldn't really matter cause I'm just giving it one course
#I tried it with all courses now groq allow it cause now i got 2310 instead of 7183 so it's *3 smaller

input tockens : 2233


In [ ]:
#6 I didn't test it cause I m not using openAI api key so I just followed the lesson
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback
calls=0
def search(query: str) :
    """
    You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
    """
    global calls
    boost_dict = {'content': 3.0}
    calls += 1
    return chunked_index.search(query,num_results=5,boost_dict=boost_dict)

agent_tools = Tools()
agent_tools.add_tool(search)   


chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt="You're a course teaching assistant your job is to answer questions in a polite way",
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

print("number of calls",calls)
